In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision
from torchvision.datasets import CIFAR10

In [ ]:
# Datasets & Dataloaders

from torch.utils.data import DataLoader
import torchvision.transforms as transforms

# Convert images to scale(0,1) & normalise(-1,1)   (Converts pixel values from [0,255] to [0,1] and then normalizes them to [-1,1], why [-1,1] bcoz, centering the data around 0 will lead to faster and more stable gradient-based training which we want.
transform = transforms.Compose([ # this compose fnx takes the list of transformations to apply.
    transforms.ToTensor(), # converts to tensor and scales (0, 1)
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))  # for RGB 3 times 0.5, which shifts image values from [0,1] → [-1,1]
])

# CIFAR10 already consists of train dataset fnx
train_dataset = CIFAR10(root="./data", train=True, download=True, transform=transform)
test_dataset = CIFAR10(root="./data", train=False, download=True, transform=transform)


In [ ]:
train_dataloader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=64)

# Build CNN

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        # Convolution layers
        self.conv_layers = nn.Sequential(
            # convolution layer 1
            nn.Conv2d(3, 32, kernel_size=3, padding=1), # i/p = 3, feature_map(filter) = 32
            nn.ReLU(),
            nn.MaxPool2d(2, 2), # kernel size = 2, stride = 2
            # convolution layer 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            # convolution layer 3
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
        )

        # Fully connected layers
        self.fc_layers = nn.Sequential(
            nn.Linear(4*4*128, 256),  # check notes why this
            nn.ReLU(),

            nn.Linear(256, 10),
        )
    # forward prop...
    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1)  # flattening
        x = self.fc_layers(x)
        return x

In [ ]:
model = CNN()

# loss & optimizers
criterion = nn.CrossEntropyLoss()  # bcoz it is a multi-class classification
optimizer = optim.Adam(model.parameters())

In [ ]:
# Train CNN

epochs = 10
for epoch in range(epochs):
    model.train()
    epoch_training_loss = 0.0

    for xb, yb in train_dataloader: # xb is images and yb is o/p
        optimizer.zero_grad()
        
        outputs = model(xb)  # forward prop..
        loss = criterion(outputs, yb)  # loss computation
        loss.backward() # backward prop...
        optimizer.step() # update the params

        epoch_training_loss += loss.item()
        
    training_loss = epoch_training_loss / len(train_dataloader)
    print(f"Epoch: {epoch+1} / {epochs} & Loss: {training_loss}")

In [ ]:
# Evaluate our CNN

model.eval()

total = 0
correct = 0

with torch.no_grad():
    for xb, yb in test_dataloader:
        outputs = model(xb)
        _, predicted = torch.max(outputs, 1)

        correct += (predicted == yb).sum().item()
        total += yb.size(0)
print("Accuracy:", correct / total)